# 🗺️ Cartographer — Deep Research Agent

> *"Every great discovery begins with an unexplored question. Cartographer maps the web so you don't have to."*

## Architecture
```
Quest → [Planner] → [Explorer] → [Critic] ──(gaps?)──► [Explorer]
                                           └──(done)──► [Writer] → Treasure Map
```

**Nodes:**
- **Planner** — Decomposes the Quest into 3–5 Waypoints (sub-queries)
- **Explorer** — Parallel Tavily searches per Waypoint
- **Critic** — LLM-as-judge scores coverage (0–10), identifies Uncharted Zones
- **Writer** — Synthesizes the Treasure Map (cited Markdown report)

---

## 0. Setup & Environment

In [5]:
# Install dependencies (run once)
!pip3 install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [7]:
import sys
from pathlib import Path

# Add project root to path so src/ is importable
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

print('✅ Environment loaded')

✅ Environment loaded


## 1. Verify Configuration

In [8]:
import os

checks = {
    'GOOGLE_API_KEY': os.getenv('GOOGLE_API_KEY'),
    'TAVILY_API_KEY': os.getenv('TAVILY_API_KEY'),
    'LLM Provider': os.getenv('CARTOGRAPHER_LLM_PROVIDER', 'google'),
    'LLM Model': os.getenv('CARTOGRAPHER_LLM_MODEL', 'gemini-2.0-flash'),
    'Max Expeditions': os.getenv('MAX_EXPEDITIONS', '2'),
    'Critic Threshold': os.getenv('CRITIC_SCORE_THRESHOLD', '7.0'),
}

for k, v in checks.items():
    if v and 'KEY' in k:
        print(f'  ✅ {k}: ***{v[-4:]}')
    elif v:
        print(f'  ✅ {k}: {v}')
    else:
        print(f'  ❌ {k}: NOT SET')

  ✅ GOOGLE_API_KEY: ***wBwg
  ✅ TAVILY_API_KEY: ***5Cac
  ✅ LLM Provider: ollama
  ✅ LLM Model: gemma3:4b
  ✅ Max Expeditions: 2
  ✅ Critic Threshold: 7.0


## 2. LLM Factory — Model-Agnostic Setup

In [9]:
from src.llm_factory import build_llm

# Default: reads from env (CARTOGRAPHER_LLM_PROVIDER + CARTOGRAPHER_LLM_MODEL)
llm = build_llm(provider='ollama', model='gemma3:4b')
print(f'✅ LLM ready: {llm.__class__.__name__}')

# To switch provider, pass args directly:
# llm = build_llm(provider='anthropic', model='claude-3-haiku-20240307')
# llm = build_llm(provider='openai', model='gpt-4o-mini')

✅ LLM ready: ChatOllama


## 3. Individual Node Walkthroughs

Run each node in isolation to understand its behaviour before running the full graph.

### 3a. Planner Node — Chart the Territory

In [11]:
from src.nodes.planner import planner_node

QUEST = "What are the trade-offs between vector databases for production RAG systems?"

planner_out = await planner_node({'quest': QUEST})

waypoints = planner_out['waypoints']
print(f'📍 Waypoints ({len(waypoints)}):')
for i, w in enumerate(waypoints, 1):
    print(f'  {i}. {w}')

print(f'\n📋 Trace: {planner_out["trace"][0]["message"]}')

📍 Waypoints (4):
  1. Vector database performance benchmarks for RAG
  2. Cost analysis of vector database solutions for production
  3. Scalability considerations for vector databases in RAG systems
  4. Vector database query optimization strategies for RAG

📋 Trace: Charted **4 Waypoints**:
  1. Vector database performance benchmarks for RAG
  2. Cost analysis of vector database solutions for production
  3. Scalability considerations for vector databases in RAG systems
  4. Vector database query optimization strategies for RAG


### 3b. Explorer Node — Explore the Terrain

In [12]:
from src.nodes.explorer import explorer_node

explorer_state = {
    'quest': QUEST,
    'waypoints': waypoints,
    'terrain': [],
    'uncharted_zones': [],
    'expedition_count': 0,
}

explorer_out = await explorer_node(explorer_state)
terrain = explorer_out['terrain']

print(f'🔍 Found {len(terrain)} results:')
for r in terrain[:3]:  # Show first 3
    print(f'  [{r["score"]:.2f}] {r["title"]}')
    print(f'        {r["url"]}')
    print(f'        {r["content"][:120]}…\n')

🔍 Found 20 results:
  [0.92] Medium
        https://medium.com/@bijit211987/rag-vs-vectordb-2c8cb3e0ee52
        5. Decode Passages

The passage text corresponding to the retrieved vectors can then be accessed and decoded. Top releva…

  [0.90] Vector Databases for Efficient Data Retrieval in RAG: A Comprehensive Guide
        https://medium.com/@genuine.opinion/vector-databases-for-efficient-data-retrieval-in-rag-a-comprehensive-guide-dcfcbfb3aa5d
        Scalability: Vector databases efficiently manage large-scale, high-dimensional data management of different types of dat…

  [0.89] Vector Database Optimization: Reducing RAG Query Latency by 60% - Huzefa Nalkheda Wala
        https://huzefanalkhedawala.in/blog/2025/01/05/vector-database-optimization-rag
        Title: Vector Database Optimization: Reducing RAG Query Latency by 60% - Huzefa Nalkheda Wala
# Vector Database Optimiza…



### 3c. Critic Node — Verify the Map

In [13]:
from src.nodes.critic import critic_node

critic_state = {
    'quest': QUEST,
    'waypoints': waypoints,
    'terrain': terrain,
    'expedition_count': 1,
    'uncharted_zones': [],
    'coverage_score': 0.0,
}

critic_out = await critic_node(critic_state)

print(f'⚖️  Coverage Score: {critic_out["coverage_score"]:.1f}/10')
print(f'   Uncharted Zones: {critic_out["uncharted_zones"]}')
print(f'   Trace: {critic_out["trace"][0]["message"]}')

⚖️  Coverage Score: 8.5/10
   Uncharted Zones: []
   Trace: ✅ **Coverage Score: 8.5/10** — This terrain provides a reasonably comprehensive overview of vector database trade-offs for RAG systems, addressing performance benchmarks, cost analysis, scalability, and optimization strategies. However, there's a lack of deep dives into specific query optimization techniques beyond indexing and caching, and the cost analysis is limited to a few providers, missing a broader comparison. The 'uncharted zones' highlight the need for more detailed information on advanced optimization strategies and a wider range of pricing models.
No gaps detected.


### 3d. Writer Node — Draw the Map

In [14]:
from src.nodes.writer import writer_node
from IPython.display import Markdown, display

writer_state = {
    'quest': QUEST,
    'waypoints': waypoints,
    'terrain': terrain,
    'coverage_score': critic_out['coverage_score'],
    'uncharted_zones': critic_out['uncharted_zones'],
    'expedition_count': 1,
    'treasure_map': '',
    'sources': [],
    'trace': [],
}

writer_out = await writer_node(writer_state)

print(f'🗺️ Treasure Map generated ({len(writer_out["treasure_map"])} chars)')
print(f'   Sources cited: {len(writer_out["sources"])}')
print('---')
display(Markdown(writer_out['treasure_map']))

🗺️ Treasure Map generated (4650 chars)
   Sources cited: 20
---


Okay, here's a "Treasure Map" visualization based on the provided information, designed to guide you to key insights about vector databases for RAG.  This map focuses on highlighting the most important aspects and connections between the sources.

**TREASURE MAP: Vector Databases for RAG**

**(Central Image: A stylized vector database icon - a network of interconnected nodes)**

**Key Areas & Paths (Lines connecting to the central image):**

1. **Performance & Cost (Red Path):**
   * **Destination:**  `[12] Vector Database Benchmark 2026 | Top 10 Compared` (SaltTechno.ai) - *Focus: Speed & Cost Efficiency*
   * **Route:**  Follows from `[19] Top 8 vector databases speed and cost efficiency comparison` (emasterlabs.com) and `[20] Vector Database Pricing Comparison 2026: Real Cost Breakdown` (ranksquire.com) – *Key Insight:  Cost varies significantly by provider and scale.  Re-indexing costs are a major factor.*

2. **Database Options & Features (Blue Path):**
   * **Destination:** `[14] [We Tried and Tested 10 Best Vector Databases for RAG ...](https://www.zenml.io/blog/vector-databases-for-rag)` (ZenML.io) - *Focus: Overview of top databases and their capabilities.*
   * **Route:**  Connects from `[11] Best Vector Databases for RAG 2026: Top 7 Picks` (alphacorp.ai) and `[13] Best Vector Databases in 2026: A Complete Comparison Guide` (firecrawl.dev) – *Key Insight:  HNSW, IVF, PQ, CAGRA are common indexing types.*

3. **Scalability & Management (Green Path):**
   * **Destination:** `[1] [Medium](https://medium.com/@bijit211987/rag-vs-vectordb-2c8cb3e0ee52)` - *Focus: Auto-scaling and managed services*
   * **Route:**  Connects from `[15] [Mastering RAG: Choosing the Perfect Vector Database](https://galileo.ai/blog/mastering-rag-choosing-the-perfect-vector-database)` (Galileo.ai) and `[18] [Top 15 Vector Databases in 2026: A Production Guide - Medium](https://medium.com/@pratik-rupareliya/top-15-vector-databases-in-2026-a-production-decision-guide-from-100-enterprise-deployments-dd58a04f51a5)` – *Key Insight: Auto-scaling is crucial for performance and cost.*

4. **Pricing Models & Considerations (Yellow Path):**
   * **Destination:** `[16] [The Hidden Cost of Vector Database Pricing Models](https://www.actian.com/blog/databases/the-hidden-cost-of-vector-database-pricing-models)` (Actian.com) - *Focus:  Understanding the different cost components.*
   * **Route:**  Connects from `[17] [A Guide to Pinecone Pricing | Tiger Data](https://www.tigerdata.com/blog/a-guide-to-pinecone-pricing)` and `[20] [Vector Database Pricing Comparison 2026: Real Cost Breakdown](https://ranksquire.com/2026/03/04/vector-database-pricing-comparison-2026)` – *Key Insight:  Storage, queries, embeddings, and support costs vary greatly.*

5. **RAG Optimization & Production (Purple Path):**
   * **Destination:** `[9] [Mastering Vector Database Optimization for 2025](https://sparkco.ai/blog/mastering-vector-database-optimization-for-2025)` (Sparkco.ai) - *Focus:  Practical optimization techniques.*
   * **Route:** Connects from `[3] [Vector DB Optimization for RAG | Indexing, Sharding](https://apxml.com/courses/optimizing-rag-for-production/chapter-4-end-to-end-rag-performance/vector-db-optimization-rag)` and `[10] [Vector DB Optimization RAG | Scalable Management](https://apxml.com/courses/large-scale-distributed-rag/chapter-4-scalable-data-ingestion-processing-pipelines/vector-db-management-optimization-scale)` – *Key Insight:  SingleStoreDB for transactional integrity alongside vector retrieval.*


**Smaller Nodes & Connections (Branching off the main paths):**

* **[4] [Understanding RAG Part VII: Vector Databases & Indexing Strategies - MachineLearningMastery.com](https://machinelearningmastery.com/understanding-rag-part-vii-vector-databases-indexing-strategies)** – *Focus:  Indexing strategies (HNSW, etc.)*
* **[5] [How do you select a vector database for your RAG ...](https://www.linkedin.com/posts/pavan-belagatti_how-do-you-select-a-vector-database-for-your-activity-7275548230216536066-1-p9)** – *Focus:  Selection criteria*
* **[6] [Vector Databases for RAG](https://www.ibm.com/think/topics/rag-vector-database)** – *Focus:  Overview and use cases*

---

**Notes on the Map:**

*   This map is a simplified representation. The real landscape of vector databases is complex.
*   The colors represent the primary focus of each source.
*   The lines indicate the relationships and connections between the information.

Would you like me to elaborate on any specific aspect of this map, or perhaps create a more detailed version focusing on a particular area (e.g., cost comparison, indexing types)?

## 4. Full Graph Run

Now run the complete Cartographer graph end-to-end.

In [ ]:
from src.graph import cartographer
from IPython.display import Markdown, display, clear_output
import ipywidgets as widgets

QUEST = "What are the key risks and mitigation strategies for deploying LLMs in healthcare?"

initial_state = {
    'quest': QUEST,
    'waypoints': [],
    'terrain': [],
    'uncharted_zones': [],
    'coverage_score': 0.0,
    'expedition_count': 0,
    'treasure_map': '',
    'sources': [],
    'trace': [],
}

# ── Live streaming output widgets ─────────────────────────────────────────────
log_out = widgets.Output()
map_out = widgets.Output()

header = widgets.HTML('<h3>📍 Expedition Log</h3>')
map_header = widgets.HTML('<h3>📜 Treasure Map (streaming…)</h3>')

display(widgets.HBox([
    widgets.VBox([header, log_out], layout=widgets.Layout(width='40%')),
    widgets.VBox([map_header, map_out], layout=widgets.Layout(width='60%')),
]))

final_state = None

async for event in cartographer.astream_events(initial_state, version='v2'):
    kind = event['event']
    name = event.get('name', '')

    if kind == 'on_chain_end' and name in ('planner', 'explorer', 'critic', 'writer'):
        output = event.get('data', {}).get('output', {})
        for entry in output.get('trace', []):
            with log_out:
                display(Markdown(entry['message']))

    elif kind == "on_chat_model_stream" and event.get("metadata", {}).get("langgraph_node") == "writer":
        chunk = event.get('data', {}).get('chunk')
        if chunk and hasattr(chunk, 'content') and chunk.content:
            with map_out:
                print(chunk.content, end='', flush=True)

    elif kind == 'on_chain_end' and name == 'LangGraph':
        final_state = event.get('data', {}).get('output', {})

print('\n✅ Expedition complete!')


✅ Expedition complete!


In [16]:
# Final summary stats
if final_state:
    print(f'Coverage Score : {final_state.get("coverage_score", "N/A"):.1f}/10')
    print(f'Expeditions    : {final_state.get("expedition_count", 0)}')
    print(f'Sources cited  : {len(final_state.get("sources", []))}')
    print(f'Report length  : {len(final_state.get("treasure_map", ""))} chars')
    print('\nSources:')
    for s in final_state.get('sources', []):
        print(f'  [{s["citation_index"]}] {s["title"]}')
        print(f'      {s["url"]}')

Coverage Score : 8.5/10
Expeditions    : 1
Sources cited  : 20
Report length  : 1644 chars

Sources:
  [1] Bias patterns in the application of LLMs for clinical decision support: A comprehensive study
      https://arxiv.org/html/2404.15149v1
  [2] The Future of LLMs in Healthcare: 5 Clinical Use Cases
      https://healthtechmagazine.net/article/2024/07/future-llms-in-healthcare-clinical-use-cases-perfcon
  [3] A future role for health applications of large language models depends on regulators enforcing safety standards
      https://www.sciencedirect.com/science/article/pii/S2589750024001249
  [4] GitHub - healthylaife/MedLLM-Bias-Fairness-Resources: Resources related to studying bias and fairness in medical LLMs. · GitHub
      https://github.com/healthylaife/MedLLM-Bias-Fairness-Resourcses
  [5] LLMs in Healthcare | Revolutionizing Patient Care | ClearDATA
      https://cspm.cleardata.com/cleardata-blog/blog/llms-in-healthcare
  [6] Using large language models like ChatGPT in heal

## 5. Launch Gradio UI

Run this cell to open the full streaming Gradio interface.

In [ ]:
# Launch inline in notebook
import subprocess, sys
print('Launching Gradio UI at http://localhost:7860')
print('Or run standalone: python cartographer_gradio.py')

# Inline launch:
# exec(open('cartographer_gradio.py').read())

## 6. Next Steps

- `evals/eval_runner.ipynb` — Run the LLM-as-judge evaluation suite
- Swap LLM provider: change `CARTOGRAPHER_LLM_PROVIDER` in `.env`
- Tune the critic threshold: `CRITIC_SCORE_THRESHOLD` (default 7.0)
- Add LangSmith tracing: set `LANGCHAIN_TRACING_V2=true` in `.env`